# Определение просодических характеристик

Топ-15 самых важных признаков:
* Само слово: 23.3131
* has_stress: 18.8089
* next_word_pos: 12.3302
* word_length: 8.6399
* word_form: 5.9292
* words_after: 5.6466
* part_of_speech: 5.2393
* prev_word_pos: 4.8262
* semantics2: 3.9640
* genesys: 2.6516
* words_before: 2.1488
* sentence_length: 1.9985
* semantics1: 1.9372
* position_in_sentence: 1.6702
* starts_with_capital: 0.6663

In [1]:
# !pip3 install torch

In [2]:
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, mean_absolute_error
import xgboost as xgb
import json

In [3]:
class XMLPauseParser:
    def __init__(self):
        self.data = []
    
    def parse_xml(self, xml_content):
        """Парсит XML и извлекает данные о словах и паузах"""
        soup = BeautifulSoup(xml_content, 'xml')
        
        sentences = soup.find_all('sentence')
        
        for sentence in sentences:
            words_in_sentence = []
            current_pause = -1
            
            # Обрабатываем все элементы предложения
            for element in sentence.children:
                if element.name == 'word':
                    word_data = self._parse_word(element)
                    words_in_sentence.append(word_data)
                
                elif element.name == 'pause':
                    # Пауза после последнего слова
                    current_pause = int(element.get('time', -1))
                    if words_in_sentence:
                        words_in_sentence[-1]['pause_len'] = current_pause
            
            # Добавляем фразовое ударение
            if words_in_sentence:
                words_in_sentence[-1]['phrasal_stress'] = True
                
                for word in words_in_sentence:
                    self.data.append(word)
    
    def _parse_word(self, word):
        """Парсит информацию о слове"""
        original = word.get('original', '')
        
        # Декодируем HTML entities
        import html
        if original:
            content = html.unescape(original)
        else:
            content = ''
        
        # Базовые признаки слова
        features = {
            'content': content,
            'word_length': len(content),
            'has_comma': ',' in content,
            'has_dot': '.' in content,
            'has_exclamation': '!' in content,
            'has_question': '?' in content,
            'nucleus': word.get('nucleus', '0'),
            'phrasal_stress': False,
            'pause_len': -1
        }
        
        # Признаки из букв
        letters = word.find_all('letter')
        features['letter_count'] = len(letters)
        features['stressed_letter'] = any(letter.get('stress') for letter in letters)
        
        # Признаки из аллофонов
        allophones = word.find_all('allophone')
        features['allophone_count'] = len(allophones)
        
        # Признаки из dictitem
        dictitem = word.find('dictitem')
        if dictitem:
            features.update({
                'subpart_of_speech': dictitem.get('subpart_of_speech', '0'),
                'form': dictitem.get('form', '0'),
                'genesys': dictitem.get('genesys', '0'),
                'stress_dict': dictitem.get('stress_dict', '0'),
                'semantics2': dictitem.get('semantics2', '0')
            })
        else:
            features.update({
                'subpart_of_speech': '0',
                'form': '0',
                'genesys': '0',
                'stress_dict': '0',
                'semantics2': '0'
            })
        
        return features
    
    def get_training_data(self):
        """Подготавливает данные для обучения"""
        df = pd.DataFrame(self.data)
        
        # Создаем целевые переменные
        df['has_pause'] = (df['pause_len'] > 0).astype(int)
        
        categorical_columns = ['subpart_of_speech', 'form', 'genesys', 'stress_dict', 'semantics2']
        
        for col in categorical_columns:
            df[col] = df[col].astype(str)
        
        return df

In [4]:
class PausePredictor:
    def __init__(self):
        self.classifier = None
        self.regressor = None
        self.scaler = None
        self.label_encoders = {}
        self.feature_columns = []
    
    def prepare_features(self, df):
        """Подготавливает признаки для модели"""
        # Бинарные признаки
        binary_features = ['has_comma', 'has_dot', 'has_exclamation', 'has_question', 
                          'phrasal_stress', 'stressed_letter']
        
        # Числовые признаки
        numeric_features = ['word_length', 'letter_count', 'allophone_count']
        
        # Категориальные признаки для кодирования
        categorical_features = ['subpart_of_speech', 'form', 'genesys', 'stress_dict', 'semantics2']
        
        # Создаем копию данных
        features_df = df[binary_features + numeric_features + categorical_features].copy()
        
        for col in categorical_features:
            if col not in self.label_encoders:
                self.label_encoders[col] = LabelEncoder()
                features_df[col] = self.label_encoders[col].fit_transform(features_df[col].fillna('0'))
            else:
                features_df[col] = self.label_encoders[col].transform(features_df[col].fillna('0'))
        
        # Сохраняем список признаков
        self.feature_columns = binary_features + numeric_features + categorical_features
        
        return features_df
    
    def train(self, df):
        """Обучает модели"""
        # Подготавливаем признаки
        X = self.prepare_features(df)
        y_pause = df['has_pause']
        y_duration = df['pause_len']
        
        # Масштабируем признаки
        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X)
        
        # Разделяем данные
        X_train, X_test, y_pause_train, y_pause_test, y_dur_train, y_dur_test = train_test_split(
            X_scaled, y_pause, y_duration, test_size=0.2, random_state=42, stratify=y_pause
        )
        
        # Обучаем классификатор (есть пауза или нет)
        print("Обучение классификатора...")
        # XGBoost лучшие параметры: {'subsample': 0.8, 'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
        # self.classifier = RandomForestClassifier(
        self.classifier = xgb.XGBClassifier(
            n_estimators=500,
            subsample=0.8,
            max_depth=7,
            learning_rate=0.05,
            colsample_bytree=0.8,
            random_state=42
            # class_weight='balanced'
        )
        self.classifier.fit(X_train, y_pause_train)
        
        # Обучаем регрессор (длительность паузы)
        # RandomForest Regressor лучшие параметры: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_depth': 10}
        print("Обучение регрессора...")
        pause_indices = y_dur_train > 0
        if sum(pause_indices) > 0:
            self.regressor = RandomForestRegressor(
                n_estimators=100,
                min_samples_split=2,
                min_samples_leaf=4,
                max_depth=10,
                random_state=42
            )
            self.regressor.fit(X_train[pause_indices], y_dur_train[pause_indices])
        
        # Оценка качества
        y_pause_pred = self.classifier.predict(X_test)
        print("Классификация (пауза есть/нет):")
        print(classification_report(y_pause_test, y_pause_pred))
        
        if self.regressor and sum(y_dur_test > 0) > 0:
            pause_test_indices = y_dur_test > 0
            y_dur_pred = self.regressor.predict(X_test[pause_test_indices])
            mae = mean_absolute_error(y_dur_test[pause_test_indices], y_dur_pred)
            print(f"Регрессия (длительность паузы) MAE: {mae:.2f}")
    
    def predict(self, text_data):
        """Предсказывает паузы для новых данных"""
        if not self.classifier:
            raise ValueError("Модель не обучена!")
        
        # Создаем DataFrame из входных данных
        input_df = pd.DataFrame(text_data)
        
        # Подготавливаем признаки
        X = self.prepare_features(input_df)
        X_scaled = self.scaler.transform(X)
        
        # Предсказываем наличие паузы
        pause_pred = self.classifier.predict(X_scaled)
        pause_proba = self.classifier.predict_proba(X_scaled)[:, 1]
        
        # Предсказываем длительность паузы
        duration_pred = np.zeros(len(text_data))
        if self.regressor:
            # Предсказываем длительность только для слов с паузами
            pause_indices = pause_pred == 1
            if sum(pause_indices) > 0:
                duration_pred[pause_indices] = self.regressor.predict(X_scaled[pause_indices])
        
        # Форматируем результат
        result = {
            "words": []
        }
        
        for i in range(len(text_data)):
            word_result = {
                "content": text_data[i]['content'],
                "phrasal_stress": bool(text_data[i]['phrasal_stress']),
                "pause_len": int(duration_pred[i]) if pause_pred[i] == 1 else -1,
                "pause_probability": float(pause_proba[i])
            }
            result["words"].append(word_result)
        
        return result

In [5]:
def train():
    # Загрузка XML файла
    with open('/home/danya/datasets/text_to_speech/crime_and_punishment.Result.xml', 'r', encoding='utf-8') as f:
        xml_content = f.read()
    
    # Парсинг XML
    print("Парсинг XML...")
    parser = XMLPauseParser()
    parser.parse_xml(xml_content)
    
    # Получение данных для обучения
    df = parser.get_training_data()
    print(f"Загружено {len(df)} слов")
    print(f"С паузами: {sum(df['has_pause'])}")
    
    # Обучение модели
    print("\nОбучение модели...")
    predictor = PausePredictor()
    predictor.train(df)
    
    return predictor

In [6]:
predictor = train()

Парсинг XML...
Загружено 41470 слов
С паузами: 9893

Обучение модели...
Обучение классификатора...
Обучение регрессора...
Классификация (пауза есть/нет):
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      6315
           1       0.95      0.93      0.94      1979

    accuracy                           0.97      8294
   macro avg       0.97      0.96      0.96      8294
weighted avg       0.97      0.97      0.97      8294

Регрессия (длительность паузы) MAE: 84.61


In [7]:
import json
import pandas as pd
from bs4 import BeautifulSoup

def predict_pauses_from_xml(predictor, xml_file_path):
    """Предсказывает паузы для XML файла используя обученную модель"""
    
    # Парсим XML файл
    with open(xml_file_path, 'r', encoding='utf-8') as f:
        xml_content = f.read()
    
    soup = BeautifulSoup(xml_content, 'xml')
    sentences = soup.find_all('sentence')
    
    # Собираем все слова из XML
    all_words_data = []
    
    for sentence in sentences:
        for word in sentence.find_all('word'):
            word_data = _parse_word_for_prediction(word)
            all_words_data.append(word_data)
    
    # Предсказываем паузы
    prediction_result = predictor.predict(all_words_data)
    
    # Форматируем результат в нужный JSON формат
    formatted_result = {
        "words": [
            {
                "content": word_pred["content"],
                "phrasal_stress": word_pred["phrasal_stress"],
                "pause_len": word_pred["pause_len"]
            }
            for word_pred in prediction_result["words"]
        ]
    }
    
    return formatted_result

def _parse_word_for_prediction(word):
    """Парсит слово из XML для предсказания"""
    import html
    
    original = word.get('original', '')
    content = html.unescape(original) if original else ''
    
    # Признаки из букв
    letters = word.find_all('letter')
    stressed_letter = any(letter.get('stress') for letter in letters)
    
    # Признаки из аллофонов
    allophones = word.find_all('allophone')
    
    # Признаки из dictitem
    dictitem = word.find('dictitem')
    if dictitem:
        subpart_of_speech = dictitem.get('subpart_of_speech', '0')
        form = dictitem.get('form', '0')
        genesys = dictitem.get('genesys', '0')
        stress_dict = dictitem.get('stress_dict', '0')
        semantics2 = dictitem.get('semantics2', '0')
    else:
        subpart_of_speech = '0'
        form = '0'
        genesys = '0'
        stress_dict = '0'
        semantics2 = '0'
    
    sentence = word.find_parent('sentence')
    if sentence:
        words_in_sentence = sentence.find_all('word')
        is_last_word = word == words_in_sentence[-1] if words_in_sentence else False
    else:
        is_last_word = False
    
    return {
        'content': content,
        'word_length': len(content),
        'has_comma': ',' in content,
        'has_dot': '.' in content,
        'has_exclamation': '!' in content,
        'has_question': '?' in content,
        'phrasal_stress': is_last_word,
        'stressed_letter': stressed_letter,
        'letter_count': len(letters),
        'allophone_count': len(allophones),
        'subpart_of_speech': subpart_of_speech,
        'form': form,
        'genesys': genesys,
        'stress_dict': stress_dict,
        'semantics2': semantics2
    }

In [8]:
result = predict_pauses_from_xml(predictor, '/home/danya/datasets/text_to_speech/Test_Procody.xml')

In [9]:
with open('Test_Procody.json', 'w') as f:
    json.dump([result], f, ensure_ascii=False, indent=4)